In [0]:
# STEP 1: Store your NEW Claude API key in Databricks Secrets
# Replace 'your-new-claude-api-key-here' with the actual key you just created

# Uncomment and run this line once, then comment it back out for security:
# dbutils.secrets.put(scope="my-api-keys", key="claude-api-key", string_value="your-new-claude-api-key-here")

# After storing, verify it works:
try:
    test_key = dbutils.secrets.get(scope="my-api-keys", key="claude-api-key")
    print(f"✅ API key stored successfully!")
    print(f"   Key starts with: {test_key[:8]}...")
    print(f"   Key length: {len(test_key)} characters")
    
    if not test_key.startswith("sk-ant-"):
        print("\n⚠️  WARNING: Key doesn't start with 'sk-ant-'")
        print("   Make sure you copied the correct key from Anthropic Console")
    else:
        print("\n✅ Key format looks correct!")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nTo store the key, uncomment the dbutils.secrets.put line above and run this cell.")

In [0]:
import anthropic

In [0]:
# STEP 2: Test the new Claude API key to find which model works
import anthropic

print("="*70)
print("TESTING NEW CLAUDE API KEY")
print("="*70)

# Get the API key from secrets
try:
    claude_api_key = dbutils.secrets.get(scope="my-api-keys", key="claude-api-key")
    print("\n✅ API key retrieved from secrets")
except Exception as e:
    print(f"\n❌ Failed to get API key: {e}")
    print("Run the previous cell first to store your API key!")
    raise

# Initialize Anthropic client
client = anthropic.Anthropic(api_key=claude_api_key)

# Test different Claude models (newest first)
test_models = [
    "claude-3-5-sonnet-20241022",  # Newest Sonnet (Oct 2024)
    "claude-3-5-haiku-20241022",   # Newest Haiku (Oct 2024)
    "claude-3-5-sonnet-20240620",  # Older Sonnet (June 2024)
    "claude-3-opus-20240229",      # Opus (Feb 2024)
    "claude-3-sonnet-20240229",    # Older Sonnet (Feb 2024)
    "claude-3-haiku-20240307"      # Older Haiku (March 2024)
]

print("\n🔍 Testing models...\n")

working_model = None
for model in test_models:
    try:
        print(f"   Testing {model}...", end=" ")
        response = client.messages.create(
            model=model,
            max_tokens=20,
            messages=[{"role": "user", "content": "Say 'Hello' if you can hear me"}]
        )
        print(f"✅ WORKS!")
        print(f"      Response: {response.content[0].text}")
        working_model = model
        break  # Found a working model, stop testing
    except anthropic.NotFoundError:
        print("❌ 404 - Model not found or no access")
    except anthropic.AuthenticationError:
        print("❌ 401 - Invalid API key")
        print("\n⚠️  Your API key is invalid. Please:")
        print("   1. Go to https://console.anthropic.com/settings/keys")
        print("   2. Create a new API key")
        print("   3. Update the key in the previous cell")
        break
    except anthropic.PermissionDeniedError:
        print("❌ 403 - No permission for this model")
    except Exception as e:
        print(f"❌ {type(e).__name__}: {str(e)[:80]}")

print("\n" + "="*70)
if working_model:
    print("🎉 SUCCESS! Your Claude API is working!")
    print("="*70)
    print(f"\n✅ Working model: {working_model}")
    print(f"\n🔧 Next step: Update table_extractor.py to use this model")
    print(f"   File: /Workspace/Users/gb.burcea@gmail.com/agentic_quality_check/src/utils/table_extractor.py")
    print(f"   Line 44: self.model_name = \"{working_model}\"")
else:
    print("❌ NO WORKING MODEL FOUND")
    print("="*70)
    print("\n🛠️ Troubleshooting:")
    print("   1. Verify your API key at: https://console.anthropic.com/settings/keys")
    print("   2. Check your account has model access in Workbench")
    print("   3. Ensure you have credits: https://console.anthropic.com/settings/billing")
    print("   4. Try creating a fresh API key")
    print("   5. Contact Anthropic support if issues persist")

In [0]:
# Restart Python to load newly installed packages
dbutils.library.restartPython()

In [0]:
# Setup dynamic paths - works for any user
import os

# Get current username dynamically
username = spark.sql("SELECT current_user()").collect()[0][0]

# Or use this alternative method:
# username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

# Build paths dynamically
PROJECT_ROOT = f"/Workspace/Users/{username}/agentic_quality_check"
SRC_PATH = f"{PROJECT_ROOT}/src"

# Add to Python path
import sys
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print(f"✅ Configured for user: {username}")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📦 Source path: {SRC_PATH}")

In [0]:
%pip install pdfplumber

In [0]:
# Test the REAL TableExtractor with Phi-3-Mini LLM
import sys
import importlib.util

# Ensure PROJECT_ROOT and SRC_PATH are defined
if 'PROJECT_ROOT' not in globals():
    username = spark.sql("SELECT current_user()").collect()[0][0]
    PROJECT_ROOT = f"/Workspace/Users/{username}/agentic_quality_check"
    SRC_PATH = f"{PROJECT_ROOT}/src"

# Add SRC_PATH to sys.path before any imports
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Load prerequisite data: parse PDF (Cell 3 logic)
if 'result' not in globals():
    # Use importlib to load pdf_parser module
    pdf_parser_spec = importlib.util.spec_from_file_location(
        "pdf_parser",
        f"{SRC_PATH}/utils/pdf_parser.py"
    )
    pdf_parser_module = importlib.util.module_from_spec(pdf_parser_spec)
    pdf_parser_spec.loader.exec_module(pdf_parser_module)
    parse_pdf = pdf_parser_module.parse_pdf
    pdf_path = "/Volumes/my_catalog/agentic_quality_check_dev/pdfs_volume/Multiplication_check.pdf"
    result = parse_pdf(pdf_path)
    print(f"PDF parsed: {result['filename']} ({len(result['headlines'])} headlines)")

# Load CSV metadata (Cell 6 logic)
if 'csv_data' not in globals() or 'csv_path' not in globals():
    # Use importlib to load csv_handler module
    csv_handler_spec = importlib.util.spec_from_file_location(
        "csv_handler",
        f"{SRC_PATH}/utils/csv_handler.py"
    )
    csv_handler_module = importlib.util.module_from_spec(csv_handler_spec)
    csv_handler_spec.loader.exec_module(csv_handler_module)
    get_csv_metadata = csv_handler_module.get_csv_metadata
    csv_path = "/Volumes/my_catalog/agentic_quality_check_dev/csvs_volume/mtc_national_pupil_characteristics_2022_to_2025.csv"
    csv_data = get_csv_metadata(csv_path)
    print(f"CSV loaded: {csv_data['filename']} ({csv_data['row_count']:,} rows)")

# Import TableExtractor
spec = importlib.util.spec_from_file_location(
    "table_extractor_module", 
    f"{SRC_PATH}/utils/table_extractor.py"
)
table_extractor_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(table_extractor_module)

TableExtractor = table_extractor_module.TableExtractor

print('\n' + '='*70)
print('Claude Powered Table Extraction Test')
print('='*70)

# Initialize extractor
# Get Claude API key from secrets
claude_api_key = dbutils.secrets.get(scope="my-api-keys", key="claude-api-key")

try:
    # Claude required for complex pivot generation (Phi-3 too weak)
    extractor = TableExtractor(use_claude=True, api_key=claude_api_key)
    print("\nModel loaded successfully!\n")
except Exception as e:
    print(f"\nModel loading failed: {type(e).__name__}: {e}")
    raise

# Get test headlines (from parsed PDF)
test_headline_keywords = ['Attainment by gender']

selected_headlines = []
for h in result['headlines']:
    for keyword in test_headline_keywords:
        if keyword.lower() in h['text'].lower():
            selected_headlines.append(h)
            print(f"📌 Selected headline: '{h['text']}' (Page {h['page']})")
            if h['paragraphs']:
                print(f"\nFull Context:")
                for i, para in enumerate(h['paragraphs'], 1):
                    print(f"   Paragraph {i}: {para}")
                print()
            break

if not selected_headlines:
    print("No matching headlines found!")
else:
    # Test extraction for each headline
    for headline in selected_headlines:
        print(f"EXTRACTING: '{headline['text']}'")
        # Run agent extraction
        extraction_result = extractor.extract_table(
            headline=headline, 
            csv_path=csv_path, 
            column_metadata=csv_data['columns']
        )
        
        print("\nEXTRACTION RESULT:")
  

        if 'error' in extraction_result:
            print(f"Error: {extraction_result['error']}")
            if 'pandas_code' in extraction_result:
                print(f"\\Generated Code (attempted):")
                print(extraction_result['pandas_code'])
        else:
            print(f"Success!")
            print(f"Confidence: {extraction_result['confidence']:.2f}")
            print(f"Rows extracted: {extraction_result['row_count']}")
            print(f"Columns: {extraction_result['columns_selected']}")

            print(f"\n LLM-Generated Pandas Code:")

            print(extraction_result['pandas_code'])
        

            print(f"\nExtracted Pivot Table:")
            import pandas as pd
            df_extracted = pd.DataFrame(extraction_result['extracted_table'])
            display(df_extracted)

            if extraction_result.get('filters_applied'):
                print(f"\nFilters Applied: {extraction_result['filters_applied']}")
            
            print("\n" + "="*70)
            print("AGENT EXTRACTION COMPLETE!")
            print("="*70)


In [0]:
%pip install anthropic